# Home Credit Default Risk: SQL Analysis

Author: Tiago Garcia Guimarães

## 1. Database Setup

I load the cleaned dataset into a SQLite database to answer business questions with SQL.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('../data/application_train_clean.csv')
conn = sqlite3.connect('../data/home_credit.db')
df.to_sql('applications', conn, if_exists='replace', index=False)

307511

## 2. First Look at the Table

I check the first rows and the total number of loans using SQL.

In [2]:
pd.read_sql('SELECT * FROM applications LIMIT 5', conn)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,DAYS_EMPLOYED_MISSING,AGE_YEARS,AGE_GROUP
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0.0,0.0,0.0,0.0,0.0,1.0,0,25.920548,"(20, 30]"
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0,45.931507,"(40, 50]"
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0,52.180822,"(50, 60]"
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,NaN,NaN,NaN,NaN,NaN,NaN,0,52.068493,"(50, 60]"
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0,54.608219,"(50, 60]"


In [3]:
pd.read_sql('SELECT COUNT(*) AS total_loans FROM applications', conn)

,total_loans
0,307511


## 3. Target Variable

I count how many loans were repaid (0) and how many defaulted (1), and calculate the overall default rate.

In [4]:
pd.read_sql('SELECT TARGET, COUNT(*) AS loans FROM applications GROUP BY TARGET', conn)

,TARGET,loans
0,0,282686
1,1,24825


In [5]:
pd.read_sql('SELECT AVG(TARGET) AS default_rate FROM applications', conn)

,default_rate
0,0.080729


## 4. Data Quality

I check how many loans have no value in DAYS_EMPLOYED (the placeholder we replaced in the EDA) and which income types they belong to.

In [6]:
pd.read_sql('SELECT COUNT(*) AS missing_days_employed FROM applications WHERE DAYS_EMPLOYED IS NULL', conn)

,missing_days_employed
0,55374


In [7]:
pd.read_sql('SELECT NAME_INCOME_TYPE, COUNT(*) AS loans FROM applications WHERE DAYS_EMPLOYED IS NULL GROUP BY NAME_INCOME_TYPE ORDER BY loans DESC', conn)

,NAME_INCOME_TYPE,loans
0,Pensioner,55352
1,Unemployed,22


## 5. Default Rate Analysis

I compare the default rate across age groups and income types, and also show how many loans each group has.

In [8]:
pd.read_sql('SELECT AGE_GROUP, COUNT(*) AS loans, ROUND(AVG(TARGET) * 100, 2) AS default_rate_pct FROM applications GROUP BY AGE_GROUP ORDER BY AGE_GROUP', conn)

,AGE_GROUP,loans,default_rate_pct
0,"(20, 30]",45021,11.46
1,"(30, 40]",82308,9.58
2,"(40, 50]",76541,7.65
3,"(50, 60]",68062,6.13
4,"(60, 70]",35579,4.92


In [9]:
pd.read_sql('SELECT NAME_INCOME_TYPE, COUNT(*) AS loans, ROUND(AVG(TARGET) * 100, 2) AS default_rate_pct FROM applications GROUP BY NAME_INCOME_TYPE ORDER BY default_rate_pct DESC', conn)

,NAME_INCOME_TYPE,loans,default_rate_pct
0,Maternity leave,5,40.00
1,Unemployed,22,36.36
2,Working,158774,9.59
3,Commercial associate,71617,7.48
4,State servant,21703,5.75
5,Pensioner,55362,5.39
6,Student,18,0.00
7,Businessman,10,0.00


In [10]:
pd.read_sql('SELECT NAME_EDUCATION_TYPE, COUNT(*) AS loans, ROUND(AVG(TARGET) * 100, 2) AS default_rate_pct FROM applications GROUP BY NAME_EDUCATION_TYPE ORDER BY default_rate_pct DESC', conn)

,NAME_EDUCATION_TYPE,loans,default_rate_pct
0,Lower secondary,3816,10.93
1,Secondary / secondary special,218391,8.94
2,Incomplete higher,10277,8.48
3,Higher education,74863,5.36
4,Academic degree,164,1.83


In [11]:
pd.read_sql('SELECT NAME_INCOME_TYPE, COUNT(*) AS loans, ROUND(AVG(TARGET) * 100, 2) AS default_rate_pct FROM applications GROUP BY NAME_INCOME_TYPE HAVING COUNT(*) >= 1000 ORDER BY default_rate_pct DESC', conn)

,NAME_INCOME_TYPE,loans,default_rate_pct
0,Working,158774,9.59
1,Commercial associate,71617,7.48
2,State servant,21703,5.75
3,Pensioner,55362,5.39


## 6. Employment Flag

I check whether loans with no employment value (DAYS_EMPLOYED_MISSING) have a different default rate from the rest.

In [12]:
pd.read_sql('SELECT DAYS_EMPLOYED_MISSING, COUNT(*) AS loans, ROUND(AVG(TARGET) * 100, 2) AS default_rate_pct FROM applications GROUP BY DAYS_EMPLOYED_MISSING', conn)

,DAYS_EMPLOYED_MISSING,loans,default_rate_pct
0,0,252137,8.66
1,1,55374,5.40


## 7. Key Findings

1. The overall default rate is 8.07% (24,825 of 307,511 loans), matching the EDA.
2. Default rate falls with age: 11.46% for clients aged 20 to 30 versus 4.92% for clients aged 60 to 70.
3. Default rate falls with education: 10.93% for Lower secondary versus 5.36% for Higher education. Academic degree shows 1.83%, but it has only 164 loans.
4. Among income types with at least 1,000 loans, Working has the highest default rate (9.59%) and Pensioner the lowest (5.39%).
5. Loans with no employment value (55,374, almost all pensioners) default less (5.40%) than the rest (8.66%), so DAYS_EMPLOYED_MISSING is a useful feature for the model.

## 8. Next Steps

1. Tableau: build a dashboard of default rate by age group, education and income type.
2. Machine learning: train a classification model to predict default, handling the 92/8 class imbalance.

In [13]:
conn.close()